## IMPORT LIBS

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

## Transform features for MLP

In [2]:
# Cell 2 — đổi tên để không ghi đè
train_features = np.load("features\\train_data.npy")
val_features   = np.load("features\\val_data.npy")
test_features  = np.load("features\\test_data.npy")

train_labels_raw = np.load("features\\train_labels.npy")
val_labels_raw   = np.load("features\\val_labels.npy")
test_labels_raw  = np.load("features\\test_labels.npy")

# One-hot cho model khác
encoder = OneHotEncoder(sparse_output=False)
train_labels_onehot = encoder.fit_transform(train_labels_raw.reshape(-1, 1))
val_labels_onehot   = encoder.transform(val_labels_raw.reshape(-1, 1))
test_labels_onehot  = encoder.transform(test_labels_raw.reshape(-1, 1))

# StandardScaler
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
val_features   = scaler.transform(val_features)
test_features  = scaler.transform(test_features)

## CNN MODEL

In [3]:
def buildResnet(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze 3 layer cuối để fine-tune
    for block in [model.layer2, model.layer3, model.layer4]:
        for param in block.parameters():
            param.requires_grad = True

    in_features = model.fc.in_features
    model.fc = nn.Sequential(  # type: ignore
        nn.Dropout(0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, num_classes),
    )

    return model


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Transform Images for CNN

In [4]:
class ImageDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.long)

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [6]:
# Encode labels thành số
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Load CSV ảnh
train_csv = pd.read_csv("CSVs\\train_images.csv")
val_csv = pd.read_csv("CSVs\\val_images.csv")
test_csv = pd.read_csv("CSVs\\test_images.csv")

# Fit encoder trên train, transform tất cả
le.fit(train_csv["emotion"])
train_labels = le.transform(train_csv["emotion"])
val_labels = le.transform(val_csv["emotion"])
test_labels = le.transform(test_csv["emotion"])

# Tạo Dataset
train_dataset = ImageDataset(
    train_csv["path"].tolist(), train_labels, transform=train_transform
)
val_dataset = ImageDataset(
    val_csv["path"].tolist(), val_labels, transform=val_test_transform
)
test_dataset = ImageDataset(
    test_csv["path"].tolist(), test_labels, transform=val_test_transform
)

# Tạo DataLoader
train_loader = DataLoader(
    train_dataset, batch_size=32, pin_memory=True, shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=32, pin_memory=True, shuffle=False
)
test_loader = DataLoader(
    test_dataset, batch_size=32, pin_memory=True, shuffle=False
)

print(
    f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}, Test samples: {len(test_dataset)}"
)
print("Classes:", list(le.classes_))

Train samples: 23812, Val samples: 744, Test samples: 745
Classes: ['Anger', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad']


# EVALUATE FUNCTION

In [7]:
def evaluate(loader, model, criterion):
    model.eval()

    y_true, y_pred = [], []
    total_loss = 0

    with torch.no_grad():
        for images, emotions in loader:
            images = images.to(device)
            emotions = emotions.to(device)

            output = model(images)

            loss = criterion(output, emotions)
            total_loss += loss.item()

            preds = torch.argmax(output, dim=1).cpu().numpy()

            y_true.extend(emotions.cpu().numpy())
            y_pred.extend(preds)

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)

    return avg_loss, acc, f1, prec, rec

# TRAIN / VALIDATE / TEST

## TRAIN / VALIDATE

In [8]:
EPOCH = 50
LR = 1e-4

resnet = buildResnet(num_classes=8).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet.parameters()), lr=LR
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=7
)

best_val_loss = float("inf")
patience = 5
no_improve = 0

for epoch in range(EPOCH):
    resnet.train()
    train_loss = 0

    for images, emotions in train_loader:
        images = images.to(device)
        emotions = emotions.to(device)

        optimizer.zero_grad()

        outputs = resnet(images)
        loss = criterion(outputs, emotions)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    val_loss, val_acc, val_f1, val_prec, val_rec = evaluate(
        val_loader, resnet, criterion
    )

    scheduler.step(val_loss)

    lr_now = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch + 1}/{EPOCH} | "
        f"LR {lr_now:.6f} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Acc {val_acc:.4f} | F1 {val_f1:.4f}"
    )

Epoch 1/50 | LR 0.000100 | Train Loss 1.4859 | Val Loss 1.4212 | Acc 0.5242 | F1 0.4978
Epoch 2/50 | LR 0.000100 | Train Loss 1.2581 | Val Loss 1.2956 | Acc 0.6035 | F1 0.5984
Epoch 3/50 | LR 0.000100 | Train Loss 1.1211 | Val Loss 1.2726 | Acc 0.6210 | F1 0.6145
Epoch 4/50 | LR 0.000100 | Train Loss 0.9961 | Val Loss 1.3646 | Acc 0.5901 | F1 0.5806
Epoch 5/50 | LR 0.000100 | Train Loss 0.8920 | Val Loss 1.4933 | Acc 0.5685 | F1 0.5423
Epoch 6/50 | LR 0.000100 | Train Loss 0.8023 | Val Loss 1.5472 | Acc 0.5645 | F1 0.5529
Epoch 7/50 | LR 0.000100 | Train Loss 0.7290 | Val Loss 1.5456 | Acc 0.5712 | F1 0.5634
Epoch 8/50 | LR 0.000100 | Train Loss 0.6761 | Val Loss 1.5045 | Acc 0.5981 | F1 0.5890
Epoch 9/50 | LR 0.000100 | Train Loss 0.6328 | Val Loss 1.3965 | Acc 0.6250 | F1 0.6168
Epoch 10/50 | LR 0.000100 | Train Loss 0.6085 | Val Loss 1.4543 | Acc 0.6223 | F1 0.6182
Epoch 11/50 | LR 0.000050 | Train Loss 0.5919 | Val Loss 1.3851 | Acc 0.6237 | F1 0.6220
Epoch 12/50 | LR 0.000050 | Tr

KeyboardInterrupt: 

## TEST

In [ ]:
# # Unload images and free up memory
# del train_dataset, val_dataset, test_dataset
# del train_loader, val_loader, test_loader
# del train_csv, val_csv, test_csv
# del images, emotions, output
# gc.collect()

# # Load the best model state for evaluation
# # resnet.load_state_dict(torch.load("best_model.pth"))

# # Create a new test loader with the original test data
# test_dataset_final = ImageDataset(
#     pd.read_csv("CSVs\\test_images.csv")["path"].tolist(), 
#     test_labels, 
#     transform=val_test_transform
# )
# test_loader_final = DataLoader(
#     test_dataset_final, batch_size=32, pin_memory=True, shuffle=False
# )

# # Evaluate on the test set
# test_loss, test_acc, test_f1, test_prec, test_rec = evaluate(
#     test_loader_final, resnet, criterion
# )

# print("\n--- Test Set Evaluation ---")
# print(f"Test Loss: {test_loss:.4f}")
# print(f"Test Accuracy: {test_acc:.4f}")
# print(f"Test F1-score: {test_f1:.4f}")
# print(f"Test Precision: {test_prec:.4f}")
# print(f"Test Recall: {test_rec:.4f}")

# # Clean up the final test loader
# del test_dataset_final, test_loader_final
# gc.collect()


--- Test Set Evaluation ---
Test Loss: 1.6835
Test Accuracy: 0.3208
Test F1-score: 0.2905
Test Precision: 0.4445
Test Recall: 0.3208


0